In [7]:
import pandas as pd
import shutil
import os
import numpy as np
import matplotlib.pyplot as plt
import onekey_algo.custom.components as okcomp
from onekey_algo import get_param_in_cwd

plt.rcParams['figure.dpi'] = 300
model_names = get_param_in_cwd('summary_models') + ['DLRS']
# 获取配置
task = get_param_in_cwd('task_column')
sel_m = get_param_in_cwd('sel_model')
labelf = get_param_in_cwd('label_file')
group_info = get_param_in_cwd('dataset_column') or 'group'

# 读取label文件。
labels = [task]
label_data_ = pd.read_csv(labelf)
label_data_['ID'] = label_data_['ID'].map(lambda x: f"{x}.nii.gz" if not (f"{x}".endswith('.nii.gz') or  f"{x}".endswith('.nii')) else x)
label_data_ = label_data_[['ID', group_info, task]]
label_data_ = label_data_.dropna(axis=0)

ids = label_data_['ID']
print(label_data_.columns)
label_data = label_data_
label_data

Index(['ID', 'group', 'label'], dtype='object')


,ID,group,label
0,0.nii.gz,train,1
1,1.nii.gz,train,1
2,2.nii.gz,train,1
3,3.nii.gz,train,1
4,4.nii.gz,train,1
...,...,...,...
386,05_256.nii.gz,test,1
387,05_266.nii.gz,test,1
388,05_272.nii.gz,test,1
389,05_274.nii.gz,test,1


# DLRS、DL+clinic、RS+clinic模型构建

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.metrics import brier_score_loss, accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from onekey_algo.custom.components.comp1 import normalize_df, merge_results
from sklearn.svm import SVC
from onekey_algo.custom.components.delong import delong_roc_test
from onekey_algo.custom.components.comp1 import draw_matrix
from onekey_algo.custom.components.metrics import NRI, IDI
from onekey_algo.custom.components.comp1 import plot_DCA
from onekey_algo.custom.components.comp1 import draw_calibration
from onekey_algo.custom.components import stats
from onekey_algo.custom.components.metrics import analysis_pred_binary
from onekey_algo.custom.components import nomogram


# ===================== 新增工具函数：Wilson CI for binary rate metrics =====================
def wilson_ci(success, total, conf=0.95):
    """Wilson score interval for sensitivity/specificity/ppv/npv/accuracy"""
    if total == 0:
        return np.nan, np.nan
    p = success / total
    z = norm.ppf((1 + conf) / 2)
    denominator = 1 + z**2 / total
    centre = (p + z**2 / (2*total)) / denominator
    margin = z * np.sqrt(p*(1-p)/total + z**2/(4*total**2)) / denominator
    return centre - margin, centre + margin

def metric_wilson_ci(y_true, y_pred_bin):
    tn = np.sum((y_true==0)&(y_pred_bin==0))
    fp = np.sum((y_true==0)&(y_pred_bin==1))
    fn = np.sum((y_true==1)&(y_pred_bin==0))
    tp = np.sum((y_true==1)&(y_pred_bin==1))
    total = len(y_true)
    acc_low, acc_high = wilson_ci(tp+tn, total)
    sens_low, sens_high = wilson_ci(tp, tp+fn)
    spec_low, spec_high = wilson_ci(tn, tn+fp)
    ppv_low, ppv_high = wilson_ci(tp, tp+fp) if (tp+fp)>0 else (np.nan, np.nan)
    npv_low, npv_high = wilson_ci(tn, tn+fn) if (tn+fn)>0 else (np.nan, np.nan)
    return {
        "Acc_95CI": f"{acc_low:.4f}-{acc_high:.4f}",
        "Sen_95CI": f"{sens_low:.4f}-{sens_high:.4f}",
        "Spe_95CI": f"{spec_low:.4f}-{spec_high:.4f}",
        "PPV_95CI": f"{ppv_low:.4f}-{ppv_high:.4f}",
        "NPV_95CI": f"{npv_low:.4f}-{npv_high:.4f}",
    }

def f1_bootstrap_ci(y_true, y_pred_bin, n_boot=1000, seed=123):
    """F1-score 95%CI by bootstrap percentile method"""
    np.random.seed(seed)
    n = len(y_true)
    boot_f1 = []
    for _ in range(n_boot):
        idx = np.random.choice(np.arange(n), size=n, replace=True)
        boot_f1.append(f1_score(y_true[idx], y_pred_bin[idx]))
    ci_low = np.percentile(boot_f1, 2.5)
    ci_high = np.percentile(boot_f1, 97.5)
    return f"{ci_low:.4f}-{ci_high:.4f}"

# ===================== 新增：校准指标 + bootstrap CI =====================
def calibration_metrics(y_true, prob, n_boot=1000, seed=123):
    np.random.seed(seed)
    lr_cal = LogisticRegression()
    lr_cal.fit(prob.reshape(-1,1), y_true)
    slope = lr_cal.coef_[0][0]
    intercept = lr_cal.intercept_[0]
    brier = brier_score_loss(y_true, prob)
    boot_slope = []
    boot_inter = []
    boot_brier = []
    n = len(y_true)
    for _ in range(n_boot):
        idx = np.random.choice(np.arange(n), size=n, replace=True)
        yb = y_true[idx]
        pb = prob[idx]
        lr_b = LogisticRegression()
        lr_b.fit(pb.reshape(-1,1), yb)
        boot_slope.append(lr_b.coef_[0][0])
        boot_inter.append(lr_b.intercept_[0])
        boot_brier.append(brier_score_loss(yb, pb))
    slope_ci = np.percentile(boot_slope, [2.5,97.5])
    inter_ci = np.percentile(boot_inter, [2.5,97.5])
    brier_ci = np.percentile(boot_brier, [2.5,97.5])
    return {
        "Calibration_in_the_large": intercept,
        "Calibration_intercept_CI95": f"{inter_ci[0]:.4f}-{inter_ci[1]:.4f}",
        "Calibration_slope": slope,
        "Calibration_slope_CI95": f"{slope_ci[0]:.4f}-{slope_ci[1]:.4f}",
        "Brier_score": brier,
        "Brier_CI95": f"{brier_ci[0]:.4f}-{brier_ci[1]:.4f}"
    }

# ===================== 新增：DCA 净收益计算与最高净收益阈值范围 =====================
def dca_net_benefit(y_true, prob, thresholds):
    y_true = np.asarray(y_true)
    prob = np.asarray(prob)
    n = len(y_true)
    prev = y_true.mean()
    nb = []
    for pt in thresholds:
        pred = (prob > pt).astype(int)
        tp = np.sum((y_true == 1) & (pred == 1))
        fp = np.sum((y_true == 0) & (pred == 1))
        nb.append(tp/n - (fp/n)*(pt/(1-pt)))
    nb_all = [prev - (1-prev)*(pt/(1-pt)) for pt in thresholds]
    nb_none = np.zeros(len(thresholds))
    return np.array(nb), np.array(nb_all), np.array(nb_none)

def _contiguous_ranges(x, mask):
    ranges, start = [], None
    for i, m in enumerate(mask):
        if m and start is None:
            start = i
        elif not m and start is not None:
            ranges.append((x[start], x[i-1]))
            start = None
    if start is not None:
        ranges.append((x[start], x[-1]))
    return ranges

def _fmt_ranges(ranges, nd=3):
    if not ranges:
        return ""
    return "; ".join(f"{a:.{nd}f}-{b:.{nd}f}" for a, b in ranges)

def extract_dca_best_ranges(y_true, probs, model_names,
                            thresholds=None, peak_hold_tol=0.90, nd=3):
    if thresholds is None:
        thresholds = np.linspace(0.01, 0.99, 100)
    thresholds = np.asarray(thresholds)
    nb_all_models = np.array([dca_net_benefit(y_true, p, thresholds)[0] for p in probs])

    records = []
    for k, mname in enumerate(model_names):
        nb = nb_all_models[k]
        peak_idx = int(np.argmax(nb))
        peak_nb, peak_thr = nb[peak_idx], thresholds[peak_idx]

        pos_ranges = _contiguous_ranges(thresholds, nb > 0)
        hold_ranges = _contiguous_ranges(thresholds, nb >= peak_nb * peak_hold_tol)
        best = (nb > 0) & (nb >= nb_all_models.max(axis=0))
        best_ranges = _contiguous_ranges(thresholds, best)

        records.append({
            "Model": mname,
            "Peak_Net_Benefit": round(float(peak_nb), nd),
            "Peak_Threshold": round(float(peak_thr), nd),
            "NB_positive_Range": _fmt_ranges(pos_ranges, nd),
            "Peak_hold_Range": _fmt_ranges(hold_ranges, nd),
            "NB_best_vs_others_Range": _fmt_ranges(best_ranges, nd),
        })
    return pd.DataFrame(records)

# ===================== 自定义函数 =====================
def categorical_nri_reclass_two_thresholds(old_prob, new_prob, thres_old, thres_new, y_true):
    old_bin = (old_prob > thres_old).astype(int)
    new_bin = (new_prob > thres_new).astype(int)

    event_idx = y_true == 1
    old_e = old_bin[event_idx]
    new_e = new_bin[event_idx]
    up_e  = np.sum((old_e==0)&(new_e==1)) / np.sum(event_idx)
    down_e= np.sum((old_e==1)&(new_e==0)) / np.sum(event_idx)

    nonevent_idx = y_true == 0
    old_ne = old_bin[nonevent_idx]
    new_ne = new_bin[nonevent_idx]
    down_ne = np.sum((old_ne==1)&(new_ne==0)) / np.sum(nonevent_idx)
    up_ne   = np.sum((old_ne==0)&(new_ne==1)) / np.sum(nonevent_idx)

    nri_val = (up_e - down_e) + (down_ne - up_ne)

    mat = np.zeros((2,2), dtype=int)
    for o,n in zip(old_bin, new_bin):
        mat[int(o), int(n)] +=1
    reclass_df = pd.DataFrame(mat,
    index=["Old_model_Neg","Old_model_Pos"],
    columns=["New_model_Neg","New_model_Pos"])
    return nri_val, reclass_df, old_bin, new_bin


def bootstrap_nri_two_threshold(y_true, prob_old, prob_new,
                               thres_old_ori, thres_new_ori,
                               n_boot=1000, seed=123):
    np.random.seed(seed)
    n_sample = len(y_true)

    nri_ori, reclass_ori, old_bin_ori, new_bin_ori = categorical_nri_reclass_two_thresholds(
        prob_old, prob_new, thres_old_ori, thres_new_ori, y_true
    )
    idi_ori, _ = IDI(prob_new, prob_old, y_true, with_p=True)

    boot_nri_list = []
    boot_idi_list = []

    for _ in range(n_boot):
        idx = np.random.choice(np.arange(n_sample), size=n_sample, replace=True)
        yb = y_true[idx]
        pold_b = prob_old[idx]
        pnew_b = prob_new[idx]

        # bootstrap 重抽样内重新估计阈值，用于估计阈值选择的不确定性（CI）
        _, _, _, _, _, _, _, _, _, _, thres_old_b = analysis_pred_binary(yb, pold_b)
        _, _, _, _, _, _, _, _, _, _, thres_new_b = analysis_pred_binary(yb, pnew_b)

        nri_b, _, _, _ = categorical_nri_reclass_two_thresholds(
            old_prob=pold_b, new_prob=pnew_b,
            thres_old=thres_old_b, thres_new=thres_new_b,
            y_true=yb
        )
        idi_b, _ = IDI(pnew_b, pold_b, yb, with_p=True)

        boot_nri_list.append(nri_b)
        boot_idi_list.append(idi_b)

    boot_nri_arr = np.array(boot_nri_list)
    boot_idi_arr = np.array(boot_idi_list)

    nri_ci = np.percentile(boot_nri_arr, [2.5, 97.5])
    idi_ci = np.percentile(boot_idi_arr, [2.5, 97.5])

    p_nri = 2 * min(np.mean(boot_nri_arr >= 0), np.mean(boot_nri_arr <= 0))
    p_idi = 2 * min(np.mean(boot_idi_arr >= 0), np.mean(boot_idi_arr <= 0))

    res_dict = {
        "NRI_point": nri_ori,
        "NRI_CI95_low": nri_ci[0],
        "NRI_CI95_high": nri_ci[1],
        "NRI_bootP": p_nri,
        "IDI_point": idi_ori,
        "IDI_CI95_low": idi_ci[0],
        "IDI_CI95_high": idi_ci[1],
        "IDI_bootP": p_idi,
        "thres_old_original": thres_old_ori,
        "thres_new_original": thres_new_ori
    }
    return res_dict
# ======================================================================

os.makedirs("./results", exist_ok=True)
os.makedirs("./img", exist_ok=True)

hosmer = []
youden = {}
calib_summary_all = []
fig_size = (5, 4)

for subset in [s for s in get_param_in_cwd('subsets', ['train', 'test'])]:
    ALL_results = None
    metric = []
    for mn in model_names[:-1]:
        r = pd.read_csv(f"./results/{mn}_{sel_m[mn]}_{subset if subset == 'train' else 'test'}.csv")
        r['ID'] = r['ID'].map(lambda x: os.path.basename(x).replace('.png', '.gz'))
        r.columns = ['ID', '-0', mn]
        if ALL_results is None:
            ALL_results = r
        else:
            ALL_results = pd.merge(ALL_results, r, on='ID', how='inner')
    Clinic = pd.read_csv('data/clinic_sel.csv')
    cnames = [c for c in Clinic.columns if c not in ['ID', 'group', 'label']]
    ALL_results = merge_results(ALL_results, Clinic[['ID'] + cnames], label_data[label_data[group_info] == subset], label_col='ID')
    ALL_results = ALL_results.dropna(axis=1)

    data_x = ALL_results[cnames + get_param_in_cwd('fusion_models')]
    data_y = ALL_results[task]
    if subset == 'train':
        model = LogisticRegression(random_state=0, penalty='none')
        model.fit(data_x, data_y)
        results = model.predict_proba(data_x)
        results = pd.DataFrame(results, index=ALL_results['ID'], columns=[f'{task}-0', f'{task}-1']).reset_index()
        results.to_csv(f'./results/Nomo_{subset}.csv', index=False, header=True)

        coef_df = pd.DataFrame({
            "Feature_name": data_x.columns.tolist(),
            "Coefficient_beta": model.coef_[0],
        })
        intercept_row = pd.DataFrame({"Feature_name": ["Intercept (β0)"], "Coefficient_beta": [model.intercept_[0]]})
        dlrs_formula_df = pd.concat([intercept_row, coef_df], ignore_index=True)
        dlrs_formula_df.to_csv("./results/DLRS_intercept_coefficients.csv", index=False)
        display(dlrs_formula_df)

    pred_column = [f'{task}-0', f'{task}-1']
    Nomo_results = pd.read_csv(f'./results/Nomo_{subset}.csv', header=0)
    Nomo_results.columns = ['ID', 'label-9', model_names[-1]]
    ALL_results = pd.merge(ALL_results, Nomo_results, on='ID', how='inner')
    gt = [np.array(ALL_results[task]) for _ in model_names]
    pred_train = [np.array(ALL_results[d]) for d in model_names]
    okcomp.comp1.draw_roc(gt, pred_train, labels=model_names, title=f'Cohort {subset} ROC', auto_point=False)
    plt.savefig(f'img/{subset}_auc.svg')
    plt.show()

    # ========== 汇总所有Metric + Wilson CI ==========
    # 阈值策略：仅在训练集用 Youden index 确定阈值并固定；
    # 验证/测试集一律复用训练集阈值，不在本队列重新寻优。
    for mname, y, score in zip(model_names, gt, pred_train):
        # AUC 及 95%CI 不依赖阈值，仍由 analysis_pred_binary 计算；
        # 其余阈值依赖指标统一按 thres 重算，保证 train/test 口径一致。
        _, auc, ci, _, _, _, _, _, _, _, thres_this = analysis_pred_binary(y, score)

        if subset == 'train':
            thres = thres_this          # 训练集：Youden 最优阈值
            youden[mname] = thres       # 固定，供后续队列使用
        else:
            thres = youden[mname]       # 验证/测试集：直接复用训练集 Youden 阈值

        y_bin = (score > thres).astype(int)
        tn = int(np.sum((y == 0) & (y_bin == 0)))
        fp = int(np.sum((y == 0) & (y_bin == 1)))
        fn = int(np.sum((y == 1) & (y_bin == 0)))
        tp = int(np.sum((y == 1) & (y_bin == 1)))
        acc       = (tp + tn) / (tp + fp + tn + fn)
        sens      = tp / (tp + fn)        if (tp + fn)  > 0 else np.nan
        spec      = tn / (tn + fp)        if (tn + fp)  > 0 else np.nan
        ppv       = tp / (tp + fp)        if (tp + fp)  > 0 else np.nan
        npv       = tn / (tn + fn)        if (tn + fn)  > 0 else np.nan
        precision = ppv
        recall    = sens
        f1        = 2*tp / (2*tp + fp + fn) if (2*tp + fp + fn) > 0 else np.nan

        wil       = metric_wilson_ci(y, y_bin)
        f1_ci_str = f1_bootstrap_ci(y, y_bin, n_boot=1000, seed=123)
        calib_info= calibration_metrics(y, score, n_boot=1000, seed=123)

        metric.append((
            mname, acc, auc, ci, sens, spec, ppv, npv, precision, recall, f1, thres, subset,
            wil["Acc_95CI"], wil["Sen_95CI"], wil["Spe_95CI"], wil["PPV_95CI"], wil["NPV_95CI"], f1_ci_str
        ))
        calib_row = {"Signature": mname, "Cohort": subset, **calib_info}
        calib_summary_all.append(calib_row)
    metric_ = pd.DataFrame(metric, index=None, columns=[
        'Signature', 'Accuracy', 'AUC', '95% CI_AUC',
        'Sensitivity', 'Specificity', 'PPV', 'NPV', 'Precision', 'Recall', 'F1',
        'Threshold', 'Cohort',
        'Accuracy_95CI', 'Sensitivity_95CI', 'Specificity_95CI', 'PPV_95CI', 'NPV_95CI', 'F1_95CI'
    ])
    display(metric_)

    # 绘制Delong
    plt.figure(figsize=fig_size)
    cm = np.zeros((len(model_names), len(model_names)))
    for i, mni in enumerate(model_names):
        for j, mnj in enumerate(model_names):
            if i <= j:
                cm[i][j] = np.nan
            else:
                cm[i][j] = delong_roc_test(ALL_results[task], ALL_results[mni], ALL_results[mnj])[0][0]
    cm = pd.DataFrame(cm[1:, :-1], index=model_names[1:], columns=model_names[:-1])
    draw_matrix(cm, annot=True, cmap='jet_r', cbar=True)
    plt.title(f'Cohort {subset} Delong')
    plt.savefig(f'img/{subset}_delong.svg', bbox_inches='tight')
    plt.show()

    # ===================== bootstrap计算，输出CSV金标准 =====================
    nri_p_mat = np.full((len(model_names), len(model_names)), np.nan)
    y_test = np.array(ALL_results[task])
    pair_summary = []
    for i, m_old in enumerate(model_names):
        for j, m_new in enumerate(model_names):
            if i == j:
                continue
            prob_old = np.array(ALL_results[m_old])
            prob_new = np.array(ALL_results[m_new])
            th_old = youden[m_old]
            th_new = youden[m_new]

            nri_tmp, reclass_tab, _, _ = categorical_nri_reclass_two_thresholds(
                prob_old, prob_new, th_old, th_new, y_test
            )
            reclass_tab.to_csv(f"./results/reclass_{subset}_{m_old}_VS_{m_new}.csv")

            boot_pack = bootstrap_nri_two_threshold(
                y_true=y_test,
                prob_old=prob_old,
                prob_new=prob_new,
                thres_old_ori=th_old,
                thres_new_ori=th_new,
                n_boot=1000, seed=123
            )
            nri_p_mat[i,j] = boot_pack["NRI_bootP"]
            boot_pack.update({"Model_old": m_old, "Model_new": m_new, "Cohort": subset})
            pair_summary.append(boot_pack)

    summary_df = pd.DataFrame(pair_summary)
    summary_df.to_csv(f"./results/NRI_IDI_bootstrap_summary_{subset}.csv", index=False)
    display(summary_df)

    # ---------------- 热图从CSV读取，保证符号完全对齐 ----------------
    csv_path = f"./results/NRI_IDI_bootstrap_summary_{subset}.csv"
    df_csv = pd.read_csv(csv_path)

    cm_nri = np.zeros((len(model_names), len(model_names)))
    for i,m_old in enumerate(model_names):
        for j,m_new in enumerate(model_names):
            if i==j:
                cm_nri[i,j]=0.0
                continue
            row = df_csv[(df_csv["Model_old"]==m_old)&(df_csv["Model_new"]==m_new)]
            if len(row)==1:
                cm_nri[i,j] = row["NRI_point"].iloc[0]
    plt.figure(figsize=fig_size)
    df_nri = pd.DataFrame(cm_nri, index=model_names, columns=model_names)
    draw_matrix(df_nri, annot=True, cmap='jet_r', cbar=True)
    plt.title(f'Cohort {subset} NRI')
    plt.savefig(f'img/NRI_each_cohort_{subset}.svg', bbox_inches='tight')
    plt.show()

    cm_idi = np.zeros((len(model_names), len(model_names)))
    for i,m_old in enumerate(model_names):
        for j,m_new in enumerate(model_names):
            if i==j:
                cm_idi[i,j]=0.0
                continue
            row = df_csv[(df_csv["Model_old"]==m_old)&(df_csv["Model_new"]==m_new)]
            if len(row)==1:
                cm_idi[i,j] = row["IDI_point"].iloc[0]
    plt.figure(figsize=fig_size)
    df_idi = pd.DataFrame(cm_idi, index=model_names, columns=model_names)
    draw_matrix(df_idi, annot=True, cmap='jet_r', cbar=True)
    plt.title(f'Cohort {subset} IDI')
    plt.savefig(f'img/IDI_each_cohort_{subset}.svg', bbox_inches='tight')
    plt.show()

    mat_nri_p = np.full((len(model_names), len(model_names)), np.nan)
    for i,m_old in enumerate(model_names):
        for j,m_new in enumerate(model_names):
            if i==j:
                continue
            row = df_csv[(df_csv["Model_old"]==m_old)&(df_csv["Model_new"]==m_new)]
            if len(row)==1:
                mat_nri_p[i,j] = row["NRI_bootP"].iloc[0]
    plt.figure(figsize=fig_size)
    df_nri_p = pd.DataFrame(mat_nri_p, index=model_names, columns=model_names)
    draw_matrix(df_nri_p, annot=True, cmap='jet_r', cbar=True)
    plt.title(f'Cohort {subset} NRI pvalue')
    plt.savefig(f'img/NRI_pvalue_each_cohort_{subset}.svg', bbox_inches='tight')
    plt.show()

    mat_idi_p = np.full((len(model_names), len(model_names)), np.nan)
    for i,m_old in enumerate(model_names):
        for j,m_new in enumerate(model_names):
            if i==j:
                continue
            row = df_csv[(df_csv["Model_old"]==m_old)&(df_csv["Model_new"]==m_new)]
            if len(row)==1:
                mat_idi_p[i,j] = row["IDI_bootP"].iloc[0]
    plt.figure(figsize=fig_size)
    df_idi_p = pd.DataFrame(mat_idi_p, index=model_names, columns=model_names)
    draw_matrix(df_idi_p, annot=True, cmap='jet_r', cbar=True)
    plt.title(f'Cohort {subset} IDI pvalue')
    plt.savefig(f'img/IDI_pvalue_each_cohort_{subset}.svg', bbox_inches='tight')
    plt.show()

    # DCA
    plot_DCA([ALL_results[model_name] for model_name in model_names],
             ALL_results[task], title=f'Cohort {subset} DCA', labels=model_names,
             y_min=-0.15, remap=False)
    plt.savefig(f'img/{subset}_dca.svg')
    plt.show()

    dca_thr = np.linspace(0.01, 0.99, 100)
    dca_best_df = extract_dca_best_ranges(
        y_true=np.array(ALL_results[task]),
        probs=[np.array(ALL_results[m]) for m in model_names],
        model_names=model_names,
        thresholds=dca_thr,
    )
    dca_best_df.to_csv(f"./results/DCA_best_threshold_range_{subset}.csv", index=False)
    display(dca_best_df)

    # Calibration
    draw_calibration(pred_scores=pred_train, n_bins=5, remap=False,
                     y_test=gt, model_names=model_names)
    plt.title(f'Cohort {subset} Calibration')
    plt.savefig(f'img/{subset}_cali.svg')
    plt.show()

    # HLTest
    hl_res = [stats.hosmer_lemeshow_test(y_true, y_pred, bins=10, remap=True)
              for fn, y_true, y_pred in zip(model_names, gt, pred_train)]
    hosmer.append(hl_res)

    if subset == 'train':
        ALL_results = ALL_results.round(decimals=2)
        n = nomogram.risk_nomogram(ALL_results, result=task, save_name='img/nomogram.png',
                                    columns=list(data_x.columns), width=8000, height=5000,
                                    x_range='0.05,0.35,0.65,0.95')
        display(n)

calib_df = pd.DataFrame(calib_summary_all)
calib_df.to_csv("./results/calibration_summary_all.csv", index=False)
display(calib_df)

hl_df = pd.DataFrame(hosmer, columns=model_names)
hl_df['Cohort'] = [s for s in get_param_in_cwd('subsets', ['train', 'test'])]
hl_df.to_csv("./results/hosmer_lemeshow_summary.csv", index=False)
display(hl_df)

# 模型对比

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm
from itertools import combinations

def compute_midrank(x):
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5*(i + j - 1)
        i = j
    T2 = np.empty(N, dtype=float)
    T2[J] = T + 1
    return T2


def fastDeLong(predictions_sorted_transposed, label_1_count):
    m = label_1_count
    n = predictions_sorted_transposed.shape[1] - m
    positive_examples = predictions_sorted_transposed[:, :m]
    negative_examples = predictions_sorted_transposed[:, m:]
    k = predictions_sorted_transposed.shape[0]

    tx = np.empty([k, m], dtype=float)
    ty = np.empty([k, n], dtype=float)
    tz = np.empty([k, m + n], dtype=float)
    for r in range(k):
        tx[r, :] = compute_midrank(positive_examples[r, :])
        ty[r, :] = compute_midrank(negative_examples[r, :])
        tz[r, :] = compute_midrank(predictions_sorted_transposed[r, :])
    aucs = tz[:, :m].sum(axis=1) / m / n - float(m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx[:, :]) / n
    v10 = 1.0 - (tz[:, m:] - ty[:, :]) / m
    sx = np.cov(v01)
    sy = np.cov(v10)
    delongcov = sx / m + sy / n
    return aucs, delongcov


def delong_roc_test(y_true, y_pred1, y_pred2):
    y_true = np.array(y_true)
    y_pred1 = np.array(y_pred1)
    y_pred2 = np.array(y_pred2)
    order = (-y_true).argsort()
    y_true_sorted = y_true[order]
    p1 = y_pred1[order]
    p2 = y_pred2[order]
    pos_count = int(np.sum(y_true_sorted == 1))
    preds = np.stack([p1, p2])
    aucs, cov = fastDeLong(preds, pos_count)
    auc1, auc2 = aucs[0], aucs[1]
    delta_auc = auc2 - auc1
    var_delta = cov[1,1] + cov[0,0] - 2*cov[0,1]
    se_delta = np.sqrt(var_delta)
    z = delta_auc / se_delta
    pval = 2 * (1 - norm.cdf(abs(z)))
    ci_low = delta_auc - 1.96 * se_delta
    ci_high = delta_auc + 1.96 * se_delta
    return auc1, auc2, delta_auc, ci_low, ci_high, pval


# ===================== 新增：单模型 AUC + DeLong 95%CI =====================
def auc_with_ci_single(y_true, score, conf=0.95):
    """单模型 AUC 及其 DeLong 法 95% CI"""
    y_true = np.asarray(y_true)
    score = np.asarray(score, dtype=float)
    order = (-y_true).argsort()
    ys = y_true[order]
    s = score[order]
    pos_count = int(np.sum(ys == 1))
    preds = np.atleast_2d(s)
    aucs, cov = fastDeLong(preds, pos_count)
    auc = float(np.atleast_1d(aucs)[0])
    cov2 = np.atleast_2d(cov)
    var = float(cov2[0, 0])
    se = np.sqrt(var) if var > 0 else np.nan
    z = norm.ppf((1 + conf) / 2)
    return auc, auc - z * se, auc + z * se


# ===================== 新增：Wilson 95%CI（比例类指标）=====================
def wilson_ci(success, total, conf=0.95):
    """Wilson score interval，适用于 Sens/Spe/PPV/NPV/Accuracy"""
    if total == 0:
        return np.nan, np.nan
    p = success / total
    z = norm.ppf((1 + conf) / 2)
    denominator = 1 + z**2 / total
    centre = (p + z**2 / (2*total)) / denominator
    margin = z * np.sqrt(p*(1-p)/total + z**2/(4*total**2)) / denominator
    return centre - margin, centre + margin


# ===================== 新增：Youden 最优阈值 =====================
def youden_threshold(y_true, score):
    """在本队列内按 Youden 指数（Sens+Spe-1）最大化确定阈值，判定用 score >= threshold"""
    y = np.asarray(y_true)
    s = np.asarray(score, dtype=float)
    thresholds = np.unique(s)
    best_t, best_j = float(np.max(s)), -np.inf
    for t in thresholds:
        pred = (s >= t).astype(int)
        tp = np.sum((y == 1) & (pred == 1))
        fn = np.sum((y == 1) & (pred == 0))
        tn = np.sum((y == 0) & (pred == 0))
        fp = np.sum((y == 0) & (pred == 1))
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        j = sens + spec - 1
        if j > best_j:
            best_j, best_t = j, float(t)
    return best_t, best_j


# ===================== 新增：F1 的 bootstrap 95%CI =====================
def f1_bootstrap_ci(y_true, y_pred_bin, n_boot=1000, seed=123, conf=0.95):
    y_true = np.asarray(y_true).astype(int)
    y_pred_bin = np.asarray(y_pred_bin).astype(int)
    rng = np.random.RandomState(seed)
    n = len(y_true)
    vals = []
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        yb, pb = y_true[idx], y_pred_bin[idx]
        tp = np.sum((yb == 1) & (pb == 1))
        fp = np.sum((yb == 0) & (pb == 1))
        fn = np.sum((yb == 1) & (pb == 0))
        if (2*tp + fp + fn) == 0:
            continue
        vals.append(2*tp / (2*tp + fp + fn))
    lo = np.percentile(vals, (1-conf)/2*100)
    hi = np.percentile(vals, (1+conf)/2*100)
    return lo, hi


# ===================== 新增：单模型完整指标 =====================
def model_metrics(y_true, score, threshold=None, conf=0.95):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(score, dtype=float)
    if threshold is None:
        threshold, _ = youden_threshold(y, s)
    pred = (s >= threshold).astype(int)

    tp = int(np.sum((y == 1) & (pred == 1)))
    fn = int(np.sum((y == 1) & (pred == 0)))
    tn = int(np.sum((y == 0) & (pred == 0)))
    fp = int(np.sum((y == 0) & (pred == 1)))
    n_pos, n_neg = tp + fn, tn + fp

    acc = (tp + tn) / len(y)
    sens = tp / n_pos if n_pos > 0 else np.nan
    spec = tn / n_neg if n_neg > 0 else np.nan
    ppv = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    npv = tn / (tn + fn) if (tn + fn) > 0 else np.nan
    f1 = 2*tp / (2*tp + fp + fn) if (2*tp + fp + fn) > 0 else np.nan

    auc, auc_lo, auc_hi = auc_with_ci_single(y, s, conf)

    acc_lo, acc_hi = wilson_ci(tp + tn, len(y), conf)
    sen_lo, sen_hi = wilson_ci(tp, n_pos, conf)
    spe_lo, spe_hi = wilson_ci(tn, n_neg, conf)
    ppv_lo, ppv_hi = wilson_ci(tp, tp + fp, conf) if (tp + fp) > 0 else (np.nan, np.nan)
    npv_lo, npv_hi = wilson_ci(tn, tn + fn, conf) if (tn + fn) > 0 else (np.nan, np.nan)
    f1_lo, f1_hi = f1_bootstrap_ci(y, pred, conf=conf)

    def f(v):
        return round(float(v), 3) if not np.isnan(v) else np.nan
    def ci(lo, hi):
        return f"{lo:.3f}-{hi:.3f}" if not (np.isnan(lo) or np.isnan(hi)) else "NA"

    return {
        "Threshold": f(threshold),
        "AUC": f(auc), "AUC_95CI": ci(auc_lo, auc_hi),
        "Accuracy": f(acc), "Accuracy_95CI": ci(acc_lo, acc_hi),
        "Sensitivity": f(sens), "Sensitivity_95CI": ci(sen_lo, sen_hi),
        "Specificity": f(spec), "Specificity_95CI": ci(spe_lo, spe_hi),
        "PPV": f(ppv), "PPV_95CI": ci(ppv_lo, ppv_hi),
        "NPV": f(npv), "NPV_95CI": ci(npv_lo, npv_hi),
        "F1": f(f1), "F1_95CI": ci(f1_lo, f1_hi),
        "TP": tp, "FP": fp, "TN": tn, "FN": fn,
        "N_positive": n_pos, "N_negative": n_neg,
    }


# ===================== 主程序，这里你只改csv文件名 =====================
df = pd.read_csv(r'C:\Users\Administrator\Desktop\问题回复\测试集模型对比\label验证集.csv')
#下面名字必须与表格列名一致
label = df["label"].values
prob_RS = df["prob_RS"].values
prob_DLS = df["prob_DLS"].values
prob_DLRS = df["prob_DLRS"].values
prob_RSClinc = df["prob_RSClinc"].values
prob_DLClinc = df["prob_DLClinc"].values
prob_DLRSclinc = df["prob_DLRSclinc"].values
prob_Clinc = df["prob_Clinc"].values
#可以无需修改，作为结果表上的列名
models = {
    "RS-only": prob_RS,
    "DLS-only": prob_DLS,
    "DL-RS": prob_DLRS,
    "RS-Clinc": prob_RSClinc,
    "DL-Clinc": prob_DLClinc,
    "DLRS": prob_DLRSclinc,
    "Clinic": prob_Clinc
}
name_list = list(models.keys())
prob_list = list(models.values())

res = []
for (i,j) in combinations(range(len(name_list)), 2):
    n1 = name_list[i]
    n2 = name_list[j]
    p1 = prob_list[i]
    p2 = prob_list[j]
    auc1, auc2, delta, cil, cih, p = delong_roc_test(label, p1, p2)
    res.append({
        "对比组1": n1,
        "对比组2": n2,
        "组1_AUC": round(auc1,3),
        "组2_AUC": round(auc2,3),
        "配对差值_ΔAUC": round(delta,3),
        "ΔAUC_95CI下限": round(cil,3),
        "ΔAUC_95CI上限": round(cih,3),
        "DeLong_P值": round(p,3)
    })

res_df = pd.DataFrame(res)
print(res_df)
res_df.to_csv("各个模型对比.csv", index=False)

# ===================== 新增：各模型 AUC / Sens / Spe / PPV / NPV / F1 / Acc + 95%CI =====================
# 说明：阈值默认在本队列内按 Youden 指数最大化确定；若需改用固定阈值，
# 把下面 THRESHOLDS 字典填上 {模型名: 阈值} 即可（其余为 None 的仍走 Youden）。
THRESHOLDS = {
    "RS-only": None,
    "DLS-only": None,
    "DL-RS": None,
    "RS-Clinc": None,
    "DL-Clinc": None,
    "DLRS": None,
    "Clinic": None,
}

metric_rows = []
for mname, p in models.items():
    m = model_metrics(label, p, threshold=THRESHOLDS.get(mname, None))
    metric_rows.append({"Signature": mname, **m})

metric_df = pd.DataFrame(metric_rows)
cols = ["Signature", "Threshold", "AUC", "AUC_95CI",
        "Accuracy", "Accuracy_95CI", "Sensitivity", "Sensitivity_95CI",
        "Specificity", "Specificity_95CI", "PPV", "PPV_95CI",
        "NPV", "NPV_95CI", "F1", "F1_95CI",
        "TP", "FP", "TN", "FN", "N_positive", "N_negative"]
metric_df = metric_df[cols]
print(metric_df)
metric_df.to_csv("各模型指标汇总.csv", index=False, encoding="utf-8-sig")

# 混淆矩阵单独一份（每模型 2x2，[[TN,FP],[FN,TP]]）
metric_df[["Signature", "Threshold", "TP", "FP", "TN", "FN", "N_positive", "N_negative"]] \
    .to_csv("各模型混淆矩阵.csv", index=False, encoding="utf-8-sig")


# 模型辅助与不辅助阅读者的指标对比

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.stats.contingency_tables import mcnemar
from scipy.stats import norm

def get_diag_metrics(pred, gt):
    pred = np.array(pred, dtype=int)
    gt = np.array(gt, dtype=int)
    tp = np.sum((pred == 1) & (gt == 1))
    tn = np.sum((pred == 0) & (gt == 0))
    fp = np.sum((pred == 1) & (gt == 0))
    fn = np.sum((pred == 0) & (gt == 1))
    acc = (tp + tn) / len(gt)
    sen = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    spe = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    bal_acc = (sen + spe) / 2
    return acc, sen, spe, bal_acc

def metric_diff_ci(x_unassist, x_assist, gt):
    """计算Sen/Spe/Acc/BalancedAcc 差值+CI"""
    acc_u, sen_u, spe_u, bal_u = get_diag_metrics(x_unassist, gt)
    acc_a, sen_a, spe_a, bal_a = get_diag_metrics(x_assist, gt)

    # 配对四格表，用于差值CI计算
    a = np.sum((x_unassist == 1) & (x_assist == 1))
    b = np.sum((x_unassist == 1) & (x_assist == 0))
    c = np.sum((x_unassist == 0) & (x_assist == 1))
    d = np.sum((x_unassist == 0) & (x_assist == 0))
    n = a + b + c + d
    z = norm.ppf(0.975)

    def diff_ci(val_u, val_a, discord_b, discord_c):
        diff = val_a - val_u
        se = np.sqrt((discord_b + discord_c - (discord_b - discord_c)**2 / n) / (n**2))
        low = diff - z * se
        high = diff + z * se
        return diff, low, high

    diff_acc, acc_low, acc_high = diff_ci(acc_u, acc_a, b, c)
    diff_sen, sen_low, sen_high = diff_ci(sen_u, sen_a, b, c)
    diff_spe, spe_low, spe_high = diff_ci(spe_u, spe_a, b, c)
    diff_bal, bal_low, bal_high = diff_ci(bal_u, bal_a, b, c)

    res_dict = {
        "acc_u":acc_u, "acc_a":acc_a, "diff_acc":diff_acc, "acc_ci":[acc_low, acc_high],
        "sen_u":sen_u, "sen_a":sen_a, "diff_sen":diff_sen, "sen_ci":[sen_low, sen_high],
        "spe_u":spe_u, "spe_a":spe_a, "diff_spe":diff_spe, "spe_ci":[spe_low, spe_high],
        "bal_u":bal_u, "bal_a":bal_a, "diff_bal":diff_bal, "bal_ci":[bal_low, bal_high]
    }
    return res_dict

def run_reader_analysis(vec_unassist, vec_assist, vec_gt, name):
    # McNemar检验（判读结果是否变化）
    table = [
        [np.sum((vec_unassist==1) & (vec_assist==1)), np.sum((vec_unassist==1) & (vec_assist==0))],
        [np.sum((vec_unassist==0) & (vec_assist==1)), np.sum((vec_unassist==0) & (vec_assist==0))]
    ]
    mcnemar_p = mcnemar(table, exact=True).pvalue
    met = metric_diff_ci(vec_unassist, vec_assist, vec_gt)
    row = {
        "Reader":name,
        "Acc_unassist": round(met["acc_u"],4),
        "Acc_assist": round(met["acc_a"],4),
        "Acc_diff(effect size)": round(met["diff_acc"],4),
        "Acc_95CI": f"[{met['acc_ci'][0]:.4f}, {met['acc_ci'][1]:.4f}]",

        "Sen_unassist": round(met["sen_u"],4),
        "Sen_assist": round(met["sen_a"],4),
        "Sen_diff(effect size)": round(met["diff_sen"],4),
        "Sen_95CI": f"[{met['sen_ci'][0]:.4f}, {met['sen_ci'][1]:.4f}]",

        "Spe_unassist": round(met["spe_u"],4),
        "Spe_assist": round(met["spe_a"],4),
        "Spe_diff(effect size)": round(met["diff_spe"],4),
        "Spe_95CI": f"[{met['spe_ci'][0]:.4f}, {met['spe_ci'][1]:.4f}]",

        "BalancedAcc_unassist": round(met["bal_u"],4),
        "BalancedAcc_assist": round(met["bal_a"],4),
        "BalancedAcc_diff(effect size)": round(met["diff_bal"],4),
        "BalancedAcc_95CI": f"[{met['bal_ci'][0]:.4f}, {met['bal_ci'][1]:.4f}]",

        "McNemar_P": round(mcnemar_p,4)
    }
    return row

# =========主程序=========
df = pd.read_csv(r'C:\Users\Administrator\Desktop\问题回复\human.csv')
# 这里把ground_truth换成你表格里真值那一列的真实列名！
gt_col = "ground_truth"
df = df.dropna(subset=[
    gt_col,
    "junior_unassisted","junior_assisted",
    "senior_unassisted","senior_assisted",
    "urologist_unassisted","urologist_assisted"
])
gt = df[gt_col].values

result_list = []
result_list.append(run_reader_analysis(df["junior_unassisted"].values, df["junior_assisted"].values, gt, "Junior radiologist"))
result_list.append(run_reader_analysis(df["senior_unassisted"].values, df["senior_assisted"].values, gt, "Senior radiologist"))
result_list.append(run_reader_analysis(df["urologist_unassisted"].values, df["urologist_assisted"].values, gt, "Urologist"))

res_df = pd.DataFrame(result_list)
print(res_df)
res_df.to_csv("reader_performance_result.csv", index=False, encoding="utf-8-sig")
print("\n✅ 结果已保存 reader_performance_result.csv")
